# Granger-causality analysis negative control
This notebook runs a Granger-causality analysis as a negative control for the main analysis pipeline. The goal is to confirm that any causal relationships detected between time series in the real data (DVG NGS read counts and plaque assay measurements) reflect genuine temporal dependencies rather than artifacts of the test itself. To build the negative control, we destroy the temporal structure of the measured data in two complementary ways: (1) bootstrapped min–max sampling, where each timepoint is replaced by a value drawn uniformly between the observed minimum and maximum across series at that timepoint, and (2) per-series shuffling, where the values within each individual time series are randomly permuted across time. Both procedures preserve the marginal distribution of the original data while eliminating any real temporal ordering, so a well-behaved Granger test should return few or no significant causal links. The plaque assay series is then concatenated back in, the data is log10-transformed and made stationary, and the same Granger pipeline is applied — letting us quantify the false-positive rate and use it as a baseline against the results from the unshuffled data.

**Structure.** The two controls (`shuffled` and `random`) are analysed **separately, end to end**: stationarity, Granger matrices, BH correction, classification, summaries, model fits and swarmplots all run per dataset. Every intermediate result lives in the `neg` registry (`neg[dataset][key][lag]`).

**Table of Contents**
- [Granger-causality analysis negative control](#Granger-causality-analysis-negative-control)
- [Data Preparation](#Data-Preparation)
  - [Create the two negative-control datasets](#Create-the-two-negative-control-datasets)
    - [Register both datasets](#Register-both-datasets)
  - [Log10 transformation](#Log10-transformation)
  - [Stationarity](#Stationarity)
- [Granger-causality test](#Granger-causality-test)
  - [Make Granger-causality matrix](#Make-Granger-causality-matrix)
    - [p-value distributions](#p-value-distributions)
  - [Summarize](#Summarize)
    - [Time series with Granger labels](#Time-series-with-Granger-labels)
- [Plot fitted models](#Plot-fitted-models)
  - [Fitting and plotting](#Fitting-and-plotting)
    - [all](#all)
    - [corrected](#corrected)
  - [plot single candidates](#plot-single-candidates)
- [Swarmplots](#Swarmplots)
  - [Per-dataset swarmplots](#Per-dataset-swarmplots)
- [Swarmplots of real data vs negative control](#Swarmplots-of-real-data-vs-negative-control)
    - [Control SSR summary](#Control-SSR-summary)
- [Group sizes with increasing lag](#Group-sizes-with-increasing-lag)
- [SSR with increasing lag](#SSR-with-increasing-lag)



In [ ]:
from utils.dip_utils import *
from utils.dip_visuals import *
from utils.granger import (
    calculate_critical_pval_bh,
    correct_p_values_bh_separately,
    create_summary_df,
    granger_labels,
    granger_causation_matrix_fixed_lag,
    make_stationary,
)
from utils.granger_prediction import (
    plot_restricted_only,
    plot_single_candidate,
    run_granger_prediction,
)
from utils.metrics import (
    better_percentage,
    calculate_weighted_mae,
    calculate_weighted_mape,
    cliffs_delta,
    cohens_d,
    dtw_mape,
    dtw_pearson,
    find_local_extrema,
    get_pval_symbol,
    mase,
    nrsme,
    pairwise_mwu_cliffs,
    test_statistical_difference,
)
from utils.plotting import (
    get_corrected_color,
    get_corrected_color_v2,
    get_main_color,
    granger_label_color_map,
    lighten_color,
    make_performance_swarmplot,
    make_performance_swarmplot_with_stats,
)

import matplotlib.pyplot as plt
import numpy as np
import pelz_datasets
import seaborn as sns

from statsmodels.tsa.stattools import adfuller

import warnings
warnings.filterwarnings('ignore')


In [ ]:
plt.rcParams['font.size'] = 20
plt.rcParams['font.family'] = 'Arial'

# Data Preparation

In [ ]:
# ----------------------------- CONFIG ---------------------------------------
import os

output_prefix = 'data/outputs/plus1_log10_random_shuffled'
os.makedirs(output_prefix, exist_ok=True)
os.makedirs(f'{output_prefix}/plots', exist_ok=True)

# The two negative controls, analysed SEPARATELY end-to-end.
DATASETS = ['shuffled', 'random']

DATASET_TITLES = {
    'shuffled': 'Shuffled',
    'random'  : 'Random (bootstrapped min-max)',
}

# Everything downstream lives in this registry: neg[dataset][key]
# -> no more leaked loop variables, no more accidental cross-talk between
#    the shuffled and the random branch.
neg = {ds: {} for ds in DATASETS}
# ----------------------------------------------------------------------------


In [ ]:
output_prefix_actual_data = 'data/outputs/plus1_log10_linear_imputation'
ts_data = pd.read_csv(f'{output_prefix_actual_data}/lin_interpolated_ts_df.csv', index_col=0)
max_fixed_lag = 4

In [ ]:
dvg_ts_data = ts_data[ts_data.columns[1:]].copy()
dvg_ts_data

## Create the two negative-control datasets


In [ ]:
n_cols =  dvg_ts_data.shape[1]
np.random.seed(42)

fig,axs = plt.subplots(figsize=(12, 5), ncols=2)
ax1,ax2 = axs[0],axs[1]

# get per-time min and max (customize if you already have these)
min_vals = dvg_ts_data.min(axis=1)
max_vals = dvg_ts_data.max(axis=1)

# generate uniformly distributed samples between min and max
ts_data_random = pd.DataFrame(
    np.random.uniform(min_vals.values[:, None], max_vals.values[:, None], size=(len(dvg_ts_data), n_cols)),
    index=dvg_ts_data.index,
    columns=[f'rand_{i+1}' for i in range(n_cols)]
)

ts_data_random.sample(min(250, n_cols), axis=1).plot(ax=ax1)
ax1.set_ylabel('NGS read counts (abs)')
ax1.legend().remove()
ax1.set_title('simulated data (bootstrapped min-max sampling)')

dvg_ts_data.plot(ax=ax2)
ax2.set_ylabel('NGS read counts (abs)')
ax2.legend().remove()
ax2.set_title('Measured data')
plt.tight_layout()
plt.show()


In [ ]:
plt.rcParams['font.size'] = 20
np.random.seed(42)
fig,axs = plt.subplots(figsize=(12, 5), ncols=2)
ax1,ax2 = axs[1],axs[0]
ts_data_shuffled = dvg_ts_data.apply(lambda x: np.random.permutation(x.values))
ts_data_shuffled.columns = [f'{col}_shuf' for col in ts_data_shuffled.columns]
dvg_ts_data.plot(ax=ax2)
ax2.set_ylabel('NGS read counts')
ax2.set_xlabel('Time post infection (days)')
ax2.set_title('Raw data\n(measured)')
ax2.legend().remove()
ts_data_shuffled.plot(ax=ax1)
ax1.set_ylabel('NGS read counts')
ax1.set_xlabel('Time post infection (days)')
ax1.legend().remove()
ax1.set_title('Shuffled negative control\n(random permutations of raw data)')
plt.tight_layout()

In [ ]:
ts_data_random = pd.concat([ts_data['plaque_assay'], ts_data_random], axis=1)
ts_data_random

In [ ]:
ts_data_shuffled = pd.concat([ts_data['plaque_assay'], ts_data_shuffled], axis=1)
ts_data_shuffled

In [ ]:
ts_data_arima = pd.concat([ts_data_shuffled,dvg_ts_data], axis=1)
# log-transform
ts_data_arima = np.log10(ts_data_arima + 1)
ts_data_arima.loc[[2.02, 2.49, 2.95, 5.99, 6.49, 6.96, 7.42, 8.42, 9.97, 10.43,
                  10.96, 11.42, 11.97, 13.99, 14.44, 14.97, 15.45, 16.48, 18.45, 18.98]] = np.nan
ts_data_arima.to_csv(f'{output_prefix}/log_transformed_arima_data.csv')
print(f"Saved linear interpolated + log10 transformed data to {output_prefix}/log_transformed_arima_data.csv")

### Register both datasets

`neg` is the single source of truth from here on: `neg[dataset]` holds the time series, the stationarity result, the Granger matrices, the summaries and the model fits for that dataset only.


In [ ]:
# Register both control datasets, persist them, and record non-zero counts.
neg['shuffled']['ts'] = ts_data_shuffled
neg['random']['ts']   = ts_data_random

for ds in DATASETS:
    D = neg[ds]
    # DVG columns only (plaque_assay is the cultivation variable, not a DVG)
    D['dvg_cols'] = [c for c in D['ts'].columns if c != 'plaque_assay']
    D['ts'].to_csv(f'{output_prefix}/{ds}_lin_interpolated_ts_data.csv')
    # colour reference used later in the swarmplots
    D['nonzero_counts'] = {k: v for k, v in (D['ts'] > 0).sum().items()}
    print(f"{ds:9s}: {D['ts'].shape[0]} timepoints x {len(D['dvg_cols'])} DVGs")

# kept for backwards-compatibility with any cell that still expects the merged dict
dvg2nonzero_counts = {k: v for ds in DATASETS for k, v in neg[ds]['nonzero_counts'].items()}


## Log10 transformation

In [ ]:
for ds in DATASETS:
    # log10(x + 1); avoids the deprecated DataFrame.applymap
    neg[ds]['log_ts'] = np.log10(neg[ds]['ts'] + 1)

neg['shuffled']['log_ts'].head()


## Stationarity

Time-series analysis relies on stationarity so that the models and predictions can be statistically relevant and to avoid the detection of spurious relationships.
Thus we need to pre process the imported raw data. I explored two methods to obtain stationarity:
1. The iterative method
-> repeatedly form the difference of one value at index i with its preceeding value i-1 in the input array

2. The direct method 
-> directly determine which increment is needed to get a stationary time-series (determine x if np.diff(x) gives us a stationary dataframe) (not implemented yet)

In [ ]:
for ds in DATASETS:
    D = neg[ds]
    print(f'Making {ds} data stationary...')
    D['stationarity'] = make_stationary(D['log_ts'], max_diff=10, autolag='AIC', max_lag=None)
    D['stationary_ts_df'] = D['stationarity'].df_stationary.fillna(0)
    D['diff_levels'] = D['stationarity'].diff_levels.to_dict()

    n_stat = (D['stationarity'].status == 'stationary').sum()
    n_tot  = len(D['stationarity'].status)
    print(f'  {ds}: {n_stat}/{n_tot} series stationary '
          f"({(D['stationarity'].status == 'non-stationary').sum()} non-stationary, "
          f"{(D['stationarity'].status == 'error').sum()} errors)")

    assert 'plaque_assay' in D['stationary_ts_df'].columns, \
        f'plaque_assay was dropped as non-stationary in the {ds} dataset'


In [ ]:
neg['shuffled']['stationary_ts_df'].to_csv(f'{output_prefix}/shuffled_stationary_ts_df.csv')

In [ ]:
neg['shuffled']['stationary_ts_df']

# Granger-causality test

In [ ]:
# importing the Granger-causality test from statsmodels
from statsmodels.tsa.stattools import grangercausalitytests

# assigning the string 'ssr_chi2test' to the variable 'test'
test = 'ssr_chi2test'
err_dips = {}

## Make Granger-causality matrix

How to read the matrix: column X Granger-causes row Y
i.e. X improves the forecasting performance of Y if included in an OLS model

In [ ]:
import pickle

# per-dataset result containers -- all keyed by fixed lag
for ds in DATASETS:
    neg[ds].update({
        'gc_matrix'          : {},
        'test_results'       : {},
        'gc_max_lag'         : {},
        'errors'             : {},   # was shared between both branches before
        'corrected_gc_matrix': {},
        'bh_critical_p_vals' : {},
    })


In [ ]:
for ds in DATASETS:
    D = neg[ds]
    stationary_ts_df = D['stationary_ts_df']

    for fix_lag in range(1, max_fixed_lag):
        print(f'[{ds}] Calculating Granger-causality matrix for fixed lag {fix_lag}...')

        (D['gc_matrix'][fix_lag],
         D['test_results'][fix_lag],
         D['gc_max_lag'][fix_lag],
         D['errors'][fix_lag]) = granger_causation_matrix_fixed_lag(
            stationary_ts_df,
            variables=stationary_ts_df.columns.tolist(),
            cultivation_values=['plaque_assay'],
            fixlag=fix_lag,
            maxlag=fix_lag,
        )

        D['corrected_gc_matrix'][fix_lag] = correct_p_values_bh_separately(
            D['gc_matrix'][fix_lag], cult_columns=1, alpha=0.05, debug=False,
        )

        D['bh_critical_p_vals'][fix_lag] = calculate_critical_pval_bh(
            D['gc_matrix'][fix_lag], cult_columns=1, alpha=0.05, debug=False,
        )

        n_err = len(D['errors'][fix_lag])
        if n_err:
            print(f'  [{ds}] lag {fix_lag}: {n_err} Granger test(s) failed')


In [ ]:
for ds in DATASETS:
    print(f'--- {ds}: critical p-values with BH correction ---')
    for fix_lag, crit in neg[ds]['bh_critical_p_vals'].items():
        print(f'  lag {fix_lag}: {crit}')


### p-value distributions


In [ ]:
import math

plt.rcParams.update({'font.size': 8})
n_lags = max_fixed_lag - 1

for ds in DATASETS:
    D = neg[ds]
    for direction, title in [('dvg_to_pfu', 'DVG -> PFU'), ('pfu_to_dvg', 'PFU -> DVG')]:
        cols = min(n_lags, 6)
        rows = math.ceil(n_lags / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(15, 2 * rows),
                                 constrained_layout=True, squeeze=False)
        axes = axes.flatten()

        for i, fixlag in enumerate(range(1, max_fixed_lag)):
            ax = axes[i]
            if direction == 'dvg_to_pfu':
                pvals = D['gc_matrix'][fixlag].iloc[0]        # plaque_assay row
            else:
                pvals = D['gc_matrix'][fixlag].iloc[1:, 0]    # plaque_assay column
            pvals.hist(bins=50, ax=ax)
            ax.set_title(f'lag={fixlag}')
            ax.set_xlabel('p-value')
            ax.set_ylabel('Frequency')

        for j in range(n_lags, len(axes)):
            axes[j].axis('off')

        plt.suptitle(f'{DATASET_TITLES[ds]} - p-values for Granger-causality {title}', fontsize=10)
        plt.show()


## Summarize

In [ ]:
# one read-count table per dataset (keys = that dataset's DVGs only)
for ds in DATASETS:
    D = neg[ds]
    D['read_counts_df'] = (
        D['stationary_ts_df'].T
        .loc[[c for c in D['stationary_ts_df'].columns if c != 'plaque_assay']]
        .reset_index()
        .rename(columns={'index': 'key'})
    )

neg['shuffled']['read_counts_df'].head()


In [ ]:
# Three labellings per dataset:
#   summary            -- raw p-values against a hard 0.05 cut-off
#   summary_corrected  -- BH-adjusted p-values against the same 0.05 cut-off
#   summary_bh         -- raw p-values against the BH critical value per direction
for ds in DATASETS:
    neg[ds].update({'summary': {}, 'summary_corrected': {}, 'summary_bh': {}})

for ds in DATASETS:
    D = neg[ds]
    for fixlag in range(1, max_fixed_lag):
        print(f'[{ds}] Building summary dataframes for lag {fixlag}...')

        common = dict(
            gc_test_results=D['test_results'][fixlag],
            gc_max_lag=D['gc_max_lag'][fixlag],
            diff_level_dct=D['diff_levels'],          # per-dataset, no longer merged
            readcounts_data=D['read_counts_df'],
            time_series_data=D['stationary_ts_df'],
            stat_tests=['ssr_ftest', 'ssr_chi2test', 'lrtest'],
            cultivation_values=['plaque_assay'],
        )

        D['summary'][fixlag] = create_summary_df(
            gc_matrix=D['gc_matrix'][fixlag],
            output_file=f'{output_prefix}/{ds}_gc_summary_df_lag{fixlag}.csv',
            **common,
        )

        D['summary_corrected'][fixlag] = create_summary_df(
            gc_matrix=D['corrected_gc_matrix'][fixlag],
            output_file=f'{output_prefix}/{ds}_gc_summary_df_corrected_lag{fixlag}.csv',
            **common,
        )

        D['summary_bh'][fixlag] = create_summary_df(
            gc_matrix=D['gc_matrix'][fixlag],
            bh_critical_pvals=D['bh_critical_p_vals'][fixlag],
            output_file=f'{output_prefix}/{ds}_gc_summary_df_bh_lag{fixlag}.csv',
            **common,
        )

### Time series with Granger labels


In [ ]:
# time series + granger label per lag, one table per dataset
for ds in DATASETS:
    D = neg[ds]
    stationary_df = D['stationary_ts_df'].T.copy()

    with_gc_labels = pd.DataFrame(index=stationary_df.index)
    for fixlag in range(1, max_fixed_lag):
        for cultivation_value in ['plaque_assay']:
            for _, row in D['summary'][fixlag].iterrows():
                with_gc_labels.loc[row['key'], f'lag{fixlag}_{cultivation_value}_granger_label'] = \
                    row[cultivation_value + '_granger_label']

    label_cols = [c for c in with_gc_labels.columns if c.endswith('_granger_label')]
    with_gc_labels = pd.concat([with_gc_labels, stationary_df], axis=1)
    with_gc_labels = with_gc_labels.dropna(subset=label_cols, how='all')

    D['with_gc_labels'] = with_gc_labels
    with_gc_labels.to_csv(f'{output_prefix}/{ds}_dvg_time_series_with_gc_labels.csv')
    print(f'[{ds}] labelled time series: {with_gc_labels.shape}')

neg['shuffled']['with_gc_labels'].head()


# Plot fitted models

## Fitting and plotting

### all

In [ ]:
plt.rcParams.update({'font.size': 16})

for ds in DATASETS:
    neg[ds].update({'pred': {}, 'ref': {}})

for ds in DATASETS:
    D = neg[ds]
    for fixlag in range(1, max_fixed_lag):
        print(f'[{ds}] Running Granger prediction analysis for lag {fixlag}...')
        D['pred'][fixlag], D['ref'][fixlag] = run_granger_prediction(
            summary_df=D['summary'][fixlag],
            gc_max_lag=D['gc_max_lag'][fixlag],
            gc_test_results=D['test_results'][fixlag],
            plot_dips=[],
            log_long_data=D['stationary_ts_df'],
            restricted_pred_index=D['summary'][fixlag].iloc[0].name,
            include_ssr=True,
            figsize=(5, 3),
            ylabel='log10(PFU)',
        )


### corrected

In [ ]:
for ds in DATASETS:
    neg[ds].update({'pred_corrected': {}, 'ref_corrected': {}})

for ds in DATASETS:
    D = neg[ds]
    for fixlag in range(1, max_fixed_lag):
        print(f'[{ds}] Running Granger prediction analysis with p-value correction for lag {fixlag}...')
        D['pred_corrected'][fixlag], D['ref_corrected'][fixlag] = run_granger_prediction(
            summary_df=D['summary_corrected'][fixlag],
            gc_max_lag=D['gc_max_lag'][fixlag],
            gc_test_results=D['test_results'][fixlag],
            plot_dips=[],
            log_long_data=D['stationary_ts_df'],
            restricted_pred_index=D['summary_corrected'][fixlag].iloc[0].name,
            include_ssr=True,
        )


## plot single candidates

In [ ]:
for ds in DATASETS:
    D = neg[ds]
    for fix_lag in range(1, max_fixed_lag):
        best = D['summary'][fix_lag].sort_values(by='plaque_assay_ssr_chi2test').iloc[0]
        print(f'[{ds}] lag {fix_lag}: best (lowest ssr_chi2 p-value) candidate = {best.key}')
        plot_single_candidate(
            cultivation_value_label='plaque_assay',
            dpis=D['stationary_ts_df'].index.values,
            row=best,
            figsize=(7, 5),
            xlabel='Time post infection (days)',
            ylabel='log10(PFU)',
            yscale='linear',
            opt_lag=fix_lag,
            gc_prediction_summary_df=D['pred'][fix_lag],
            gc_test_results=D['test_results'][fix_lag],
            log_long_data=D['stationary_ts_df'],
            title_prefix=f'{DATASET_TITLES[ds]}: ',
        )


In [ ]:
for ds in DATASETS:
    D = neg[ds]
    for fix_lag in range(1, max_fixed_lag):
        plot_restricted_only(
            cultivation_value_label='plaque_assay',
            dpis=D['stationary_ts_df'].index.values,
            row=D['summary'][fix_lag].sort_values(by='plaque_assay_ssr_chi2test').iloc[0],
            figsize=(7, 5),
            xlabel='Time post infection (days)',
            ylabel='log10(PFU)',
            yscale='linear',
            opt_lag=fix_lag,
            gc_prediction_summary_df=D['pred'][fix_lag],
            gc_test_results=D['test_results'][fix_lag],
            log_long_data=D['stationary_ts_df'],
            title_prefix=f'{DATASET_TITLES[ds]}: ',
        )


# Swarmplots

## Per-dataset swarmplots

Each negative control gets its own swarmplot, for raw (`all`) and BH-corrected labels.


In [ ]:
label_order = ['causing', 'bi-directional', 'caused', 'non-related']
_rank = {label: i for i, label in enumerate(label_order)}

for ds in DATASETS:
    D = neg[ds]
    for fixlag in range(1, max_fixed_lag):
        for variant in ['pred', 'pred_corrected']:
            df = D[variant][fixlag]
            df = df.sort_values(by='granger_label', key=lambda x: x.map(_rank))
            # colour reference used by the swarmplot
            df['non_zero_counts'] = df['key'].map(D['nonzero_counts'])
            D[variant][fixlag] = df


In [ ]:
DATASET_TITLES = {
    'shuffled': 'shuffled',
    'random': 'random'}

In [ ]:
plt.rcParams.update({'font.size': 10})

# one swarmplot per dataset x lag x variant -- each summary now contains ONLY
# the DVGs of its own dataset, so no included_dvgs filtering is needed.
VARIANT_SPECS = [
    ('(A)', 'all',       'pred',           'ref',           ''),
    ('(B)', 'corrected', 'pred_corrected', 'ref_corrected', ', Benjamini-Hochberg corrected'),
]



for ds in ['shuffled', 'random']:
    D = neg[ds]
    for title_prefix, variant, pred_key, ref_key, title_suffix in VARIANT_SPECS:
        for fixlag in range(1, 2):
            print(f'[{ds}/{variant}] Creating performance swarmplot for lag {fixlag}...')
            ref = D[ref_key][fixlag]
            fig, ax = make_performance_swarmplot(
                dip_forecast_summary=D[pred_key][fixlag],
                ref_performance_dct=ref,
                performance_metric='ssr',
                forecasted_value='plaque_assay',
                title=(f'{title_prefix} Granger-causality analysis on {DATASET_TITLES[ds]} DVG time series,\n'
                      f'OLS models for log10(PFU/mL+1) (lag={fixlag}){title_suffix}'),
                ylabel='SSR',
                dot_color_ref='group_norm_ssr_chi2test_pval',
                inverted_colors=True,
                upper_border=-0.05,
                ymax=ref['SSR'] * 1.5,
                ymin=80,
                p_val_pos=ref['SSR'] * 1.2,
                figsize=(6, 3),
                dotsize=3,
                linewidth=2,
                dpi=800,
                title_pad=10,
                median_halo=True,
            )
            fig.savefig(f'{output_prefix}/plots/{ds}_ssr_swarmplot_{variant}_lag{fixlag}.png',
                        dpi=800, bbox_inches='tight')
            plt.show()


# Swarmplots of real data vs negative control

Each real DVG group (`causing`, `bi-directional`, `caused`, `non-related`) is
compared against a **shuffled negative control** (same series, temporal order
destroyed, run through the identical pipeline). Points are per-DVG SSR of the
full model; the `+` marks the group median and the orange line the restricted
(PFU-only) reference.

Brackets show pairwise **Mann–Whitney U** (Bonferroni-corrected) with **Cliff's
δ** for p < 0.05, matching notebook 03. Only real-vs-control comparisons are
tested; real-vs-real is omitted because the groups are defined by the same
statistic that drives SSR (circular). Shown for both uncorrected and
BH-corrected labels.

* Loads the 4 REAL Granger groups exported by notebook 01.
* Takes the SHUFFLED negative-control DVGs (computed in this notebook),
relabels them 'shuffled', and appends them as a 5th swarm.
* Runs pairwise Mann-Whitney U (Bonferroni-corrected) + Cliff's delta.
* Draws brackets ONLY for real-vs-shuffled comparisons (the statistically
valid, non-circular test) and annotates each with Cliff's delta.
Patched make_performance_swarmplot below == nb03's, plus a `granger_label_col`
/ `key_col` shim so it consumes the native nb01/nb02 schema directly.

In [ ]:
plt.rcParams.update({'font.size': 18})

# ------------------------- CONFIG (edit here) -------------------------------
LAG                      = 1
NB01_OUTPUT_DIR          = 'data/outputs/plus1_log10_linear_imputation'  # real-data nb output_prefix
REAL_REF_CSV             = f'{NB01_OUTPUT_DIR}/ref_ols_performance_dct.csv'
REAL_LABELS              = ['causing', 'bi-directional', 'caused', 'non-related']
METRIC                   = 'ssr'
COLOR_REF                = 'group_norm_ssr_chi2test_pval'
MW_ADJUST                = 'bonferroni'   # multiple-testing correction for MWU
ALPHA                    = 0.05           # bracket shown when adjusted p < ALPHA
BRACKETS_VS_CONTROL_ONLY = True           # True => only real-vs-control brackets (avoids double-dipping)
COLOR_CONTROL_BY_LABEL   = True           # True => control dots coloured by their OWN granger label
SAVE_DIR                 = f'{output_prefix}/plots'
# The control label / DVG set / x-axis order are now derived per dataset
# ('shuffled', 'random') in the cells below, so the real data is compared
# against each negative control separately.
# ----------------------------------------------------------------------------


In [ ]:
# restricted-model reference line (same for every variant; from nb01)
ref_df  = pd.read_csv(REAL_REF_CSV)
ref_row = ref_df[ref_df['lag'] == LAG].iloc[0]
ref_dct = {'SSR': float(ref_row['SSR'])}

VARIANTS = [
    dict(tag='uncorrected',
         real_csv=f'{NB01_OUTPUT_DIR}/gc_prediction_summary_lag{LAG}.csv',
         ctrl_key='pred',
         title_tag='uncorrected labels'),
    dict(tag='corrected',
         real_csv=f'{NB01_OUTPUT_DIR}/gc_prediction_summary_corrected_lag{LAG}.csv',
         ctrl_key='pred_corrected',
         title_tag='BH-corrected labels'),
]

PLOT_DATA = {}   # (dataset, tag) -> dict(combined, mw, delta, order, title_tag)

for ds in DATASETS:                       # <-- real data vs EACH control separately
    D = neg[ds]
    control_label = ds
    order = REAL_LABELS + [control_label]

    for V in VARIANTS:
        # 1) real groups (nb01 export) -- colour label == x-axis label
        real = pd.read_csv(V['real_csv'])
        real = real[real['granger_label'].isin(REAL_LABELS)].copy()
        real['color_label'] = real['granger_label']

        # 2) negative control (this notebook, in memory) -> relabel to the control name
        ctrl = D[V['ctrl_key']][LAG].copy()
        ctrl['color_label'] = ctrl['granger_label'] if COLOR_CONTROL_BY_LABEL else control_label
        ctrl['granger_label'] = control_label              # x-axis position
        ctrl['key'] = f'ctrl_{ds}__' + ctrl['key'].astype(str)   # keep hue keys unique
        _m = ctrl[COLOR_REF].max()                        # self-consistent shading
        if _m and np.isfinite(_m) and _m > 0:
            ctrl[COLOR_REF] = ctrl[COLOR_REF] / _m

        # 3) combine on the swarmplot schema
        keep = ['key', 'granger_label', 'color_label', METRIC, COLOR_REF, 'ssr_chi2test_pval']
        combined = pd.concat([real[keep], ctrl[keep]], ignore_index=True).dropna(subset=[METRIC])
        combined = combined[combined['granger_label'].isin(order)].copy()

        # 4) pairwise MWU (Bonferroni) + Cliff's delta  <-- the slow part
        mw, delta = pairwise_mwu_cliffs(
            combined.rename(columns={'granger_label': 'label'}),
            value_col=METRIC, group_col='label', label_order=order, mw_adjust=MW_ADJUST)
        if BRACKETS_VS_CONTROL_ONLY:
            rmask = mw.index != control_label
            cmask = mw.columns != control_label
            mw.loc[rmask, cmask] = 1.0     # suppress real-vs-real brackets (circular)

        PLOT_DATA[(ds, V['tag'])] = dict(combined=combined, mw=mw, delta=delta,
                                          order=order, title_tag=V['title_tag'])
        print(f'prepared: {ds} / {V["tag"]}  (n={len(combined)})')


In [ ]:
PLOT_DATA[('shuffled', 'uncorrected')]['combined'].groupby('granger_label')['ssr_chi2test_pval'].agg(min_pval='min', max_pval='max').reset_index()

In [ ]:
PLOT_VARIANTS = ['uncorrected', 'corrected']   # e.g. ['corrected'] to redraw just one
PLOT_DATASETS = DATASETS                       # e.g. ['shuffled'] to redraw just one
title_dct = {'uncorrected': 
              f"(A) Sum of squared residuals (SSR) of OLS model fits (lag=1), grouped by Granger-causality", 
            'corrected': 
              "(B) Sum of squared residuals (SSR) of OLS model fits (lag=1), Benjamini-Hochberg-corrected"}

figs = {}


plt.rcParams.update({'font.size': 16})
#for ds in PLOT_DATASETS:
for ds in ['shuffled', 'random']:
    for tag in PLOT_VARIANTS:
        P = PLOT_DATA[(ds, tag)]
        fig, ax, PLOT_DATA[(ds, tag)]['plot_df'] = make_performance_swarmplot_with_stats(
            P['combined'], ref_performance_dct=ref_dct,
            performance_metric=METRIC, forecasted_value='plaque_assay',
            title=title_dct[tag],
            ylabel='SSR', dot_color_ref=COLOR_REF, inverted_colors=True,
            ymin=80,
            ymax=ref_dct['SSR'] * 1.3, p_val_pos=ref_dct['SSR'] * 1.1,
            figsize=(12, 8), dotsize=5, linewidth=3, dpi=800,
            stats_matrix=P['mw'], stat_test_name='mannwhitney_p',
            effect_size_matrix=P['delta'], effect_size_name='cliffs_delta',
            alpha=ALPHA, order=P['order'], plot_reference=True,
            color_label_col='color_label', title_offset=0.95)

        out_png = f"{SAVE_DIR}/real_vs_{ds}_{tag}_ssr_swarmplot_lag{LAG}.png"
        fig.savefig(out_png, dpi=800, bbox_inches='tight')
        figs[(ds, tag)] = fig
        print("saved:", out_png)


In [ ]:
PLOT_VARIANTS = ['uncorrected', 'corrected']   # e.g. ['corrected'] to redraw just one
PLOT_DATASETS = DATASETS                       # e.g. ['shuffled'] to redraw just one
title_dct = {'uncorrected': 
              f"Sum of squared residuals (SSR) of OLS model fits (lag=1), grouped by Granger-causality", 
            'corrected': 
              "Sum of squared residuals (SSR) of OLS model fits (lag=1), Benjamini-Hochberg-corrected"}

figs = {}

plt.rcParams.update({'font.size': 18})
for ds in PLOT_DATASETS:
    for tag in ['corrected']:
        P = PLOT_DATA[(ds, tag)]
        fig, ax, PLOT_DATA[(ds, tag)]['plot_df'] = make_performance_swarmplot_with_stats(
            P['combined'], ref_performance_dct=ref_dct,
            performance_metric=METRIC, forecasted_value='plaque_assay',
            title=title_dct[tag],
            ylabel='SSR', dot_color_ref=COLOR_REF, inverted_colors=True,
            ymin=80,
            ymax=200, p_val_pos=ref_dct['SSR'] * 1.1,
            figsize=(15, 6.5), dotsize=5, linewidth=3, dpi=800,
            stats_matrix=P['mw'], stat_test_name='mannwhitney_p',
            effect_size_matrix=P['delta'], effect_size_name='cliffs_delta',
            alpha=ALPHA, order=P['order'], plot_reference=True,
            color_label_col='color_label', title_offset=0.95)

        out_png = f"{SAVE_DIR}/real_vs_{ds}_{tag}_ssr_swarmplot_lag{LAG}.png"
        fig.savefig(out_png, dpi=800, bbox_inches='tight')
        figs[(ds, tag)] = fig
        print("saved:", out_png)


In [ ]:
dvg2color_df = pd.concat([PLOT_DATA[('random', 'corrected')]['plot_df'].copy(),
                          PLOT_DATA[('shuffled', 'corrected')]['plot_df'].copy()], ignore_index=True)
dvg2color_df['dvg'] = dvg2color_df['dvg'].astype(str)
dvg2color_df['dvg'] = dvg2color_df['dvg'].apply(lambda x: x.replace('ctrl_shuffled__', ''))
dvg2color_df['dvg'] = dvg2color_df['dvg'].apply(lambda x: x.replace('ctrl_random__', ''))
dvg2color_df = dvg2color_df[['dvg','label','corrected_color_name', 'corrected_color']]
dvg2color_df.to_csv(f"{SAVE_DIR}/dvg2color_shuffled_corrected.csv", index=True)
print(f"saved: {SAVE_DIR}/dvg2color_shuffled_corrected.csv")
dvg2color_df

In [ ]:
dvg2color_df[dvg2color_df['label'].isin(['causing','bi-directional','caused','non-related','shuffled'])].sort_values(by='label').to_csv(f"{output_prefix}/dvg_labels_shuffled.csv", index=False)

### Control SSR summary


In [ ]:
# median control SSR per dataset / variant
rows = []
for ds in DATASETS:
    for variant, key in [('uncorrected', 'pred'), ('corrected', 'pred_corrected')]:
        s = neg[ds][key][LAG]['ssr']
        rows.append({'dataset': ds, 'variant': variant, 'n': int(s.notna().sum()),
                      'median_ssr': s.median(), 'min_ssr': s.min(), 'max_ssr': s.max()})
control_ssr_summary = pd.DataFrame(rows)
print(control_ssr_summary.to_string(index=False))


In [ ]:
CAUSAL    = ['causing', 'bi-directional']
NONCAUSAL = ['caused', 'non-related']     # swap to ['non-related'] etc. as you like

def ssr_bounds(df, dataset, variant):
    causal    = df[df['granger_label'].isin(CAUSAL)]
    noncausal = df[df['granger_label'].isin(NONCAUSAL)]
    out = {'dataset': dataset, 'variant': variant,
           'worst_causal_ssr': np.nan, 'worst_causal_dvg': None,
           'best_noncausal_ssr': np.nan, 'best_noncausal_dvg': None}
    if len(causal) and causal['ssr'].notna().any():
        out['worst_causal_ssr'] = causal['ssr'].max()
        out['worst_causal_dvg'] = causal.loc[causal['ssr'].idxmax(), 'key']
    if len(noncausal) and noncausal['ssr'].notna().any():
        out['best_noncausal_ssr'] = noncausal['ssr'].min()
        out['best_noncausal_dvg'] = noncausal.loc[noncausal['ssr'].idxmin(), 'key']
    return out

bounds = pd.DataFrame([
    ssr_bounds(neg[ds][key][LAG], ds, variant)
    for ds in DATASETS
    for variant, key in [('uncorrected', 'pred'), ('corrected', 'pred_corrected')]
])
print(bounds.to_string(index=False))


# Group sizes with increasing lag


In [ ]:
plt.rcParams.update({'font.size': 16})

LABEL_COLORS = {'causing': 'tab:blue', 'bi-directional': 'tab:purple',
                'caused': 'tab:red', 'non-related': 'gray'}

for ds in DATASETS:
  fig, ax = plt.subplots(figsize=(8, 5))
  D = neg[ds]
  for label, color in LABEL_COLORS.items():
      sizes      = [(D['pred'][l]['granger_label'] == label).sum() for l in range(1, max_fixed_lag)]
      sizes_corr = [(D['pred_corrected'][l]['granger_label'] == label).sum() for l in range(1, max_fixed_lag)]
      ax.plot(range(1, max_fixed_lag), sizes, marker='o', color=color,
              linestyle='solid', label=label)
      ax.plot(range(1, max_fixed_lag), sizes_corr, marker='D', color=color,
              linestyle='dashed', label=f'{label} (corrected)')
  ax.set_xlabel('Fixed lag used in Granger-causality test')
  ax.set_ylabel('Number of DVGs')
  ax.set_yscale('log')
  ax.set_xticks(range(1, max_fixed_lag))
  ax.set_title(DATASET_TITLES[ds])
  ax.legend(loc='upper left', bbox_to_anchor=(1, 1), ncol=1)
  ax.set_title(f'Number of Granger-related DVGs\nacross fixed lags')
  #ax.set_title(f'[{DATASET_TITLES[ds]}] Number of Granger-related DVGs across fixed lags (solid = raw, dashed = BH-corrected)')
  plt.tight_layout()
  fig.savefig(f'{output_prefix}/plots/{ds}_granger_dvg_counts_across_lags.png', dpi=200, bbox_inches='tight')
  plt.show()


# SSR with increasing lag


In [ ]:
plt.rcParams.update({'font.size': 16})

LABEL_COLORS = {'causing': 'tab:blue', 'bi-directional': 'tab:purple',
                'caused': 'tab:red', 'non-related': 'gray'}

for variant, pred_key, ref_key, vlabel in [
        ('all',       'pred',           'ref',           'raw labels'),
        ('corrected', 'pred_corrected', 'ref_corrected', 'BH-corrected labels')]:

    for ds in DATASETS:
        fig, ax = plt.subplots(figsize=(8, 5))
        D = neg[ds]

        for fixlag in range(1, max_fixed_lag):
            ax.scatter(fixlag, D[ref_key][fixlag]['SSR'], color='tab:orange',
                       label='restricted model' if fixlag == 1 else '')

        for label, color in LABEL_COLORS.items():
            for fixlag in range(1, max_fixed_lag):
                subset = D[pred_key][fixlag]
                subset = subset[subset['granger_label'] == label]
                if len(subset):
                    ax.scatter(fixlag, subset['ssr'].median(), color=color,
                               label=label if fixlag == 1 else '', linewidth=2)

        ax.set_xlabel('Fixed lag used in Granger-causality test')
        ax.set_ylabel('Median SSR')
        ax.set_xticks(range(1, max_fixed_lag))
        ax.set_title(f'{DATASET_TITLES[ds]}\nMedian SSR vs. fixed lag ({vlabel})')
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

        plt.tight_layout()
        fig.savefig(f'{output_prefix}/plots/{ds}_median_ssr_vs_lag_{variant}.png',
                    dpi=200, bbox_inches='tight')
        plt.show()

In [ ]:
plt.rcParams.update({'font.size': 16})

for variant, pred_key, ref_key, vlabel in [
        ('all',       'pred',           'ref',           'raw labels'),
        ('corrected', 'pred_corrected', 'ref_corrected', 'BH-corrected labels')]:

    for ds in DATASETS:
        fig, ax = plt.subplots(figsize=(8, 5))
        D = neg[ds]

        for fixlag in range(1, max_fixed_lag):
            ax.scatter(fixlag, D[pred_key][fixlag]['ssr'].median(),
                       color='tab:green',
                       label='full models (median)' if fixlag == 1 else '', linewidth=2)
            ax.scatter(fixlag, D[ref_key][fixlag]['SSR'],
                       color='tab:orange',
                       label='restricted model' if fixlag == 1 else '')

        ax.set_xlabel('Fixed lag used in Granger-causality test')
        ax.set_ylabel('Median SSR')
        ax.set_xticks(range(1, max_fixed_lag))
        ax.set_title(f'{DATASET_TITLES[ds]}\nMedian SSR vs. fixed lag ({vlabel})')
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

        plt.tight_layout()
        fig.savefig(f'{output_prefix}/plots/{ds}_full_restricted_ssr_vs_lag_{variant}.png',
                    dpi=200, bbox_inches='tight')
        plt.show()